In [4]:
# 1. 原始数据：修改 openml_id，完整显示所有行、特征列和标签列。
from pathlib import Path

import numpy as np
import openml
import pandas as pd
from IPython.display import display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

openml_id = 51
dataset = openml.datasets.get_dataset(openml_id, download_data=False)
X, y, _, _ = dataset.get_data(
    dataset_format="dataframe", target=dataset.default_target_attribute
)
display(pd.concat([X, y], axis=1))
print(f"Dataset ID: {openml_id}")
print(f"样本数: {len(X)}，cluster 数（标签类别数）: {y.nunique()}，feature 数: {X.shape[1]}")


,age,sex,chest_pain,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,28.0,male,atyp_angina,130.0,132.0,f,left_vent_hyper,185.0,no,0.0,NaN,NaN,NaN,<50
1,29.0,male,atyp_angina,120.0,243.0,f,normal,160.0,no,0.0,NaN,NaN,NaN,<50
2,29.0,male,atyp_angina,140.0,NaN,f,normal,170.0,no,0.0,NaN,NaN,NaN,<50
3,30.0,female,typ_angina,170.0,237.0,f,st_t_wave_abnormality,170.0,no,0.0,NaN,NaN,fixed_defect,<50
4,31.0,female,atyp_angina,100.0,219.0,f,st_t_wave_abnormality,150.0,no,0.0,NaN,NaN,NaN,<50
5,32.0,female,atyp_angina,105.0,198.0,f,normal,165.0,no,0.0,NaN,NaN,NaN,<50
6,32.0,male,atyp_angina,110.0,225.0,f,normal,184.0,no,0.0,NaN,NaN,NaN,<50
7,32.0,male,atyp_angina,125.0,254.0,f,normal,155.0,no,0.0,NaN,NaN,NaN,<50
8,33.0,male,non_anginal,120.0,298.0,f,normal,185.0,no,0.0,NaN,NaN,NaN,<50
9,34.0,female,atyp_angina,130.0,161.0,f,normal,190.0,no,0.0,NaN,NaN,NaN,<50


Dataset ID: 51
样本数: 294，cluster 数（标签类别数）: 2，feature 数: 13


In [ ]:
# 2. 预处理数据：修改 openml_id，读取已保存的 34 个数据集之一。
# processed_features 是缺失值填补、数值缩放和类别独热编码后的特征，尚未进入 ZEUS。
openml_id = 61

with np.load(root / "data" / "embeddings" / f"openml_{openml_id}.npz") as saved:
    processed = pd.DataFrame(saved["processed_features"])
    processed.columns = [f"feature_{i}" for i in range(processed.shape[1])]
    processed["label"] = saved["labels"]

display(processed)


In [9]:
# 3. 汇总 34 个数据集：样本统计来自实验缓存，ARI 来自现有结果文件。
# LAPD 使用 known K；预测不完整的数据集，其 ARI 保留为 NaN。
kmeans = pd.read_csv(root / "results" / "zeus_openml_baseline.csv")
kmeans = kmeans.loc[kmeans["method"] == "scaled_zeus", ["openml_id", "ari"]]
kmeans = kmeans.rename(columns={"ari": "kmeans_ari"})
cpp = pd.read_csv(root / "results" / "cpp_zeus_openml.csv")
cpp = cpp[["openml_id", "ari"]].rename(columns={"ari": "cpp_ari"})
lapd = pd.read_csv(root / "results" / "lapd_stage4_audit.csv")
lapd = lapd.loc[lapd["mode"] == "known", ["openml_id", "ari"]]
lapd = lapd.rename(columns={"ari": "lapd_ari"})

rows = []
for dataset_id in sorted(kmeans["openml_id"]):
    with np.load(root / "data" / "embeddings" / f"openml_{dataset_id}.npz") as saved:
        class_counts = pd.Series(saved["labels"]).value_counts()
    rows.append({
        "openml_id": dataset_id,
        "n_samples": int(class_counts.sum()),
        "n_classes": len(class_counts),
        "mean_samples_per_class": class_counts.mean(),
        "min_samples_per_class": int(class_counts.min()),
        "max_samples_per_class": int(class_counts.max()),
    })

summary = pd.DataFrame(rows)
summary = summary.merge(kmeans, on="openml_id", how="left")
summary = summary.merge(cpp, on="openml_id", how="left")
summary = summary.merge(lapd, on="openml_id", how="left")
summary = summary.loc[summary["kmeans_ari"] < 0.5]
display(summary)
print("K-means ARI < 0.5 的数据集 ID:")
print(summary.loc[summary["kmeans_ari"] < 0.5, "openml_id"].tolist())
print(len(summary.loc[summary["n_classes"] == 2]))


,openml_id,n_samples,n_classes,mean_samples_per_class,min_samples_per_class,max_samples_per_class,kmeans_ari,cpp_ari,lapd_ari
6,51,294,2,147.000000,106,188,0.436568,0.070656,-0.008595
7,53,270,2,135.000000,120,150,0.357634,0.357634,0.192226
13,481,209,2,104.500000,75,134,0.086809,-0.036316,-0.035644
14,694,310,9,34.444444,18,41,0.333761,0.356197,0.552800
15,721,200,2,100.000000,97,103,0.432764,0.432764,0.432764
19,796,209,2,104.500000,53,156,0.139907,-0.018183,NaN
20,820,235,2,117.500000,93,142,0.288664,0.298040,-0.030547
21,840,205,2,102.500000,83,122,0.276564,0.333337,NaN
24,1495,250,2,125.000000,107,143,0.073465,0.073465,0.158819
27,1523,310,3,103.333333,60,150,0.195950,0.202234,NaN


K-means ARI < 0.5 的数据集 ID:
[51, 53, 481, 694, 721, 796, 820, 840, 1495, 1523, 40496, 40705]
9
